In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_IMPORTS

In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_AUXILIARY_FUNCTIONS

In [0]:
TABLE_ORIGIN = 'clientes_crm'

TABLE_DESTINY = 'customers_crm'

df = (
        spark.read.table(f"bronze.{TABLE_ORIGIN}")
             .withColumnsRenamed(
                                  {
                                    'id_cliente' : 'customer_id',
                                    'razao_social' : 'legal_name',
                                    'cnpj' : 'tax_id',
                                    'segmento' : 'segment',
                                    'uf_sede' : 'uf',
                                    'email_financeiro' : 'email',
                                    'data_cadastro' : 'onboarding_date',
                                    'atualizado_em' : 'updated_in'
                                  }
                                )
     )

In [0]:
df = (
        df.withColumn('tax_id', F.regexp_replace(F.col('tax_id'), r'[^0-9]', ''))
          .withColumn(
                       'onboarding_date',
                       F.date_format(
                                      F.to_date(
                                                 F.regexp_replace(F.col('onboarding_date'), r'(\d{2})/(\d{2})/(\d{4})', '$3-$2-$1'),'yyyy-MM-dd'
                                               ), 
                                      'yyyy-MM-dd'
                                    )
                     )
          .withColumn('updated_in', F.date_format(F.col('updated_in').cast('string'), 'yyyy-MM-dd'))
          .withColumn('uf', F.upper(F.col('uf')))
     )

display(df)

Let's check if 'tax_id' has only values with 14 characters as length (brazillian pattern):

In [0]:
df.groupBy(F.length(F.col('tax_id'))).count().show()

Checking the UF's values (e.g whether are real regions):

In [0]:
check_valid_federative_unit(df,'uf')

Checking potential duplicates in customers (R12):

In [0]:
check_duplicates(df, 'customer_id')

According that discussed we'll remove the duplicates:

In [0]:
check_and_remove_duplicates(df, 'customer_id')

Checking null's and blank's values:

In [0]:
display(null_or_blank(df))

The 'email' and 'segment' could be important to send charges and conduct geographic analysis, respectively.

In [0]:
# Incremental load: never replaces silver.customers_crm — appends only new customers
target = f"silver.{TABLE_DESTINY}"
key_cols = ['customer_id']

if not spark.catalog.tableExists(target):
    # first load: table does not exist yet — create it
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    print(f"✓ {target} created ({df.count()} rows)")
else:
    # append only records whose key is not present in silver yet
    df_new = df.join(spark.table(target).select(*key_cols).distinct(), on=key_cols, how='left_anti')
    n_new = df_new.count()
    if n_new:
        df_new.write.format("delta").mode("append").saveAsTable(target)
        print(f"✓ {target} appended {n_new} new row(s)")
    else:
        print(f"↷ {target}: no new records — nothing appended")

However, the main goal here not seem impacted. But regardless it, we'll sinalize this to CS team to provide adequated corrections.

In [0]:
df_quarantine = (
                   df.filter(F.col('email').isNull() | F.col('segment').isNull())
                     .withColumn(
                                  'motive',
                                  F.when(F.col('email').isNull(), 'email_null')
                                   .when(F.col('segment').isNull(), 'segment_null')
                                   .when((F.col('email').isNull()) & (F.col('segment').isNull()), 'email_null_segment_null')
                                )
                )

# Incremental load: never replaces quarantine.warning_customers — appends only new customers
target = "quarantine.warning_customers"
key_cols = ['customer_id']

if not spark.catalog.tableExists(target):
    # first load: table does not exist yet — create it
    df_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    print(f"{df_quarantine.count()} registers sent to quarantine schema")
else:
    # append only records whose key is not present in quarantine yet
    df_new = df_quarantine.join(spark.table(target).select(*key_cols).distinct(), on=key_cols, how='left_anti')
    n_new = df_new.count()
    if n_new:
        df_new.write.format("delta").mode("append").saveAsTable(target)
        print(f"{n_new} new register(s) appended to quarantine schema")
    else:
        print("No new registers for quarantine schema — nothing appended")